# 第四章：远期和期货 / Chapter 4: Forwards and Futures

## 你将学到 / What You Will Learn
- 什么是远期合约和期货合约 / What forward and futures contracts are
- 远期价格怎么算出来的 / How the forward price is derived
- 什么是"基差" / What "basis" means
- 为什么不能"无风险赚钱"（无套利原理）/ Why "risk-free profit" is not a thing (the no-arbitrage principle)

---

## 4.1 用生活例子理解 / 4.1 An Everyday Analogy

> 现在是春天，你是一个面包店老板。你知道秋天要用 1吨 小麦。
>
> It is spring. You run a bakery and you know you will need one tonne of wheat this autumn.
>
> 你担心秋天小麦涨价，于是跟农民约定：
> **"不管秋天市价多少，我以每吨 3000元 买你的小麦。"**
>
> Worried about autumn price rises, you strike a deal with a farmer:
> **"Whatever the autumn market is, I will buy your wheat at 3000 yuan per tonne."**
>
> 这个约定就是一个**远期合约**。
>
> That agreement is a **forward contract**.

- 你锁定了买入价格（不怕涨价了）/ You locked in your buy price (no fear of a rally)
- 农民锁定了卖出价格（不怕跌价了）/ The farmer locked in his sell price (no fear of a crash)
- 但如果秋天市价只有 2500，你就亏了（因为你约定了 3000）/ But if autumn market is only 2500, you lose — you already committed to 3000

**期货 / Futures** 和远期本质相同，区别是期货在交易所标准化交易，远期是两个人私下约定。

Futures and forwards are economically the same; the difference is that futures trade on exchanges in standardised sizes, while forwards are bilateral OTC deals.

## 4.2 远期价格公式 / 4.2 The Forward-Price Formula

理论上的远期价格（"公平价格"）：

The theoretical (fair) forward price:

$$\text{远期价格 / Forward} = \text{现货价格 / Spot} \times e^{\text{利率 / r} \times \text{时间 / T}}$$

直觉理解：如果你现在买现货要 100 元，资金成本 5%/年，那1年后的远期价格应该是 100 × 1.05 ≈ 105 元。多出来的 5 元就是你"占用资金"的成本。

Intuition: if spot is 100 today and your funding cost is 5% per year, the 1-year forward should be 100 × 1.05 ≈ 105. The extra 5 is simply the cost of tying up capital for a year.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, ToggleButtons
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# Unified color palette
C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done! Continue to the next cell.')

## 4.3 互动实验 / 4.3 Interactive Experiment

> 拖动滑块：
> - 提高利率 → 远期价格会怎样？
> - 延长时间 → 远期价格会怎样？
> - 如果市场报价不等于理论价格 → 能不能"无风险赚钱"？
>
> Drag the sliders:
> - Raise the rate → how does the forward price respond?
> - Extend the maturity → how does it respond?
> - If the market forward price differs from the theoretical one → can you lock in a risk-free profit?

In [ ]:
def forward_pricing(spot=100, rate=5.0, maturity=1.0):
    plt.close('all')
    r = rate / 100
    forward = spot * np.exp(r * maturity)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    # --- Left: forward price vs time ---
    ax = axes[0]
    time_grid = np.linspace(0.01, 3, 100)
    forward_curve = spot * np.exp(r * time_grid)
    ax.plot(time_grid, forward_curve, color=C_BLUE, lw=2.5)
    ax.axhline(y=spot, color=C_GREEN, ls='--', label=f'Spot {spot}')
    ax.plot(maturity, forward, 'ro', ms=12, zorder=5)
    ax.annotate(f'Forward = {forward:.2f}', xy=(maturity, forward), xytext=(maturity + 0.2, forward + 3),
                fontsize=11, fontweight='bold', color=C_RED,
                arrowprops=dict(arrowstyle='->', color=C_RED, lw=2))
    ax.set_xlabel('Maturity (years)', fontsize=11)
    ax.set_ylabel('Forward price', fontsize=11)
    ax.set_title('Forward price vs time', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # --- Middle: basis ---
    ax = axes[1]
    basis = spot - forward_curve
    ax.fill_between(time_grid, basis, 0, color=C_ORANGE, alpha=0.2)
    ax.plot(time_grid, basis, color=C_ORANGE, lw=2.5)
    ax.plot(maturity, spot - forward, 'ro', ms=10)
    ax.set_xlabel('Maturity (years)', fontsize=11)
    ax.set_ylabel('Basis', fontsize=11)
    ax.set_title(f'Basis = Spot - Forward (now = {spot - forward:.2f})', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # --- Right: arbitrage explanation ---
    ax = axes[2]
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
    ax.set_title('No-arbitrage principle', fontsize=13, fontweight='bold')
    ax.text(5, 9, f'Theoretical forward = {forward:.2f}', ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', facecolor=C_BLUE, alpha=0.2))
    ax.text(2.5, 7.2, f'If market > {forward:.0f}', ha='center', fontsize=10, color=C_RED,
            bbox=dict(boxstyle='round', facecolor='#ffe0e0'))
    for i, t in enumerate(['1. Borrow & buy spot', '2. Sell the forward', '3. Deliver at maturity -> lock profit']):
        ax.text(2.5, 5.8 - i * 0.8, t, ha='center', fontsize=9)
    ax.text(7.5, 7.2, f'If market < {forward:.0f}', ha='center', fontsize=10, color=C_GREEN,
            bbox=dict(boxstyle='round', facecolor='#e0ffe0'))
    for i, t in enumerate(['1. Short spot, deposit cash', '2. Buy the forward', '3. Deliver at maturity -> lock profit']):
        ax.text(7.5, 5.8 - i * 0.8, t, ha='center', fontsize=9)
    ax.text(5, 2.2, 'Arbitrageurs push the market back to fair value', ha='center', fontsize=11,
            fontweight='bold', color=C_PURPLE,
            bbox=dict(boxstyle='round,pad=0.3', facecolor=C_PURPLE, alpha=0.1))
    ax.text(5, 1, 'So: the forward is not a forecast, it is a no-arbitrage equilibrium',
            ha='center', fontsize=10, style='italic')

    plt.tight_layout(); plt.show()
    print(f'  Formula: F = {spot} x e^({r} x {maturity}) = {forward:.4f}')
    print(f'  The extra {forward - spot:.2f} is simply the {maturity}-year funding cost at {rate}%.')

interact(forward_pricing,
         spot=FloatSlider(value=100, min=10, max=500, step=1, description='Spot:'),
         rate=FloatSlider(value=5.0, min=0, max=20, step=0.5, description='Rate (%):'),
         maturity=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Maturity (y):'));

## 4.4 小结 / 4.4 Chapter Recap

| 概念 / Concept | 一句话 / One-liner |
|------|--------|
| 远期/期货 / Forward / Futures | 约定未来以固定价格买卖 / Lock in a buy / sell price for a future date |
| 远期价格 / Forward price | 现货价 × e^(利率×时间) / Spot × e^(rate × time) |
| 基差 / Basis | 现货价 - 远期价（到期时收敛为0）/ Spot − Forward (converges to 0 at maturity) |
| 无套利 / No arbitrage | 如果定价不对，套利者会纠正它 / If it is mispriced, arbitrageurs drag it back |

> **下一章：期权** —— 花一点小钱，买一个"选择的权利"
>
> **Next chapter: Options** — pay a small premium to buy the "right to choose".